# Healthcare Analytics: Machine Learning Model Building
This notebook trains and evaluates four machine learning algorithms on our scaled healthcare dataset:
1. Logistic Regression
2. Decision Tree Classifier
3. Random Forest Classifier
4. XGBoost Classifier


In [ ]:
import os
import pandas as pd
import numpy as np
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score


## 1. Load Scaled Partition Data


In [ ]:
X_train = pd.read_csv(os.path.join('..', 'data', 'temp', 'X_train_scaled.csv'))
X_test = pd.read_csv(os.path.join('..', 'data', 'temp', 'X_test_scaled.csv'))
y_train = pd.read_csv(os.path.join('..', 'data', 'temp', 'y_train.csv')).values.ravel()
y_test = pd.read_csv(os.path.join('..', 'data', 'temp', 'y_test.csv')).values.ravel()
print('Data Loaded Successfully. Shape:', X_train.shape)


## 2. Train Models


In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=6, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.1, random_state=42, eval_metric='logloss')
}

results = []
trained_models = {}

for name, model in models.items():
    # Train
    model.fit(X_train, y_train)
    trained_models[name] = model
    
    # Predict
    preds = model.predict(X_test)
    probs = model.predict_proba(X_test)[:, 1]
    
    # Metrics
    acc = accuracy_score(y_test, preds)
    prec = precision_score(y_test, preds, zero_division=0)
    rec = recall_score(y_test, preds)
    f1 = f1_score(y_test, preds)
    auc = roc_auc_score(y_test, probs)
    
    results.append({
        'Model': name,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-score': f1,
        'ROC-AUC': auc
    })
    print(f'Trained {name}')


## 3. Compare Models


In [ ]:
comparison_df = pd.DataFrame(results).sort_values(by='ROC-AUC', ascending=False)
comparison_df


## 4. Save Trained Models & Best Model
Choose the best model based on ROC-AUC (standard for clinical screening where we want to capture probabilities and control thresholds).


In [ ]:
os.makedirs(os.path.join('..', 'models'), exist_ok=True)
best_model_name = comparison_df.iloc[0]['Model']
best_model = trained_models[best_model_name]
print(f'Best Model: {best_model_name}')

# Save the best model
model_file = os.path.join('..', 'models', 'best_model.pkl')
joblib.dump(best_model, model_file)
print(f'Best model saved to {model_file}')

# Save all models for visualization comparisons in notebook 5
for name, model in trained_models.items():
    safe_name = name.lower().replace(' ', '_')
    joblib.dump(model, os.path.join('..', 'models', f'{safe_name}_model.pkl'))
